# IR spectra with MARS: SO3LR vs a classical force field

This notebook computes the infrared spectrum of the protonated peptide Ac-Ala$_5$-LysH$^+$ with **SO3LR Tauri-M** and with the **ff19SB** force field, and compares both with the gas-phase IRMPD experiment of [Rossi et al., J. Phys. Chem. Lett. 1, 3465 (2010)](https://doi.org/10.1021/jz101394u).

The calculations run through [MARS](https://github.com/TCPUniLU/mars), which evaluates SO3LR and classical force fields with the same commands, so the two spectra differ only in the potential. The setup follows the peptide case of the [SO3LR vibrational spectroscopy paper](https://doi.org/10.1038/s41467-026-79241-x), reduced to about half an hour on a laptop GPU.

Steps:
1. harmonic IR spectra of one optimised structure;
2. IR spectra from molecular dynamics (dipole autocorrelation), the main result;
3. comparison with experiment.

See `README.md` in this folder for the same tutorial as text, including how the ff19SB topology was built.

**Requirements**: JAX, MARS with the `mm` extra (OpenMM + ParmEd) and SO3LR:

```shell
pip install -U "jax[cuda12]<0.11"
git clone https://github.com/TCPUniLU/mars.git && cd mars && pip install -e ".[mm,viewer]" && cd ..
git clone https://github.com/general-molecular-simulations/so3lr.git && cd so3lr && pip install . && cd ..
```

Run the notebook from this folder, `examples/ir_spectra_mars/`, which contains the input files.

## Settings

In [ ]:
import shlex
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d

# Input files (in this folder)
XYZ = "ala5lysh.xyz"                       # 80 atoms, charge +1, atom order = prmtop
PRMTOP = "ala5lysh_ff19sb.prmtop"          # ff19SB topology
EXPERIMENT = "rossi2010_ala5lysh_ir.csv"   # IRMPD spectrum, 300 K

# SO3LR model, pinned to a release so the numbers stay reproducible
SO3LR_MODEL = "so3lr-tauri-m-2606"

# MD settings. 8 x 20 ps reproduces the README figure (34 min for SO3LR on a
# 4 GB laptop GPU); 4 replicas take about half as long, with a noisier spectrum.
N_REPLICAS = 8
NVT_PS = 2.0
NVE_PS = 20.0
TIMESTEP_FS = 0.5

POTENTIALS = {
    "SO3LR Tauri-M": f"--so3lr-model {SO3LR_MODEL}",
    "ff19SB": f"--potential amber --prmtop {PRMTOP}",
}
TAGS = {"SO3LR Tauri-M": "so3lr", "ff19SB": "ff19sb"}


def mars(args):
    """Run a `mars` command, print its last lines and the wall time."""
    cmd = ["mars"] + shlex.split(args)
    print("$", " ".join(cmd))
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    log = (result.stdout + result.stderr).strip().splitlines()
    print("\n".join(line for line in log[-4:] if line.strip()))
    if result.returncode != 0:
        raise RuntimeError("\n".join(log[-30:]))
    print(f"wall time: {time.time() - start:.0f} s\n")

## 1. Harmonic spectra

`mars ir` optimises the structure, computes the Hessian and the dipole derivatives numerically, by finite differences of the forces and dipoles (`--fd-hessian`, 0.01 Å displacements), and writes the stick spectrum. The intensities come from the partial charges: those SO3LR predicts at each geometry, or the fixed point charges of ff19SB.

In [ ]:
for label, potential in POTENTIALS.items():
    mars(f"ir {XYZ} {potential} --charge 1 --float64 --fd-hessian -o harm_{TAGS[label]}.dat")

## 2. Spectra from molecular dynamics

With `--md`, MARS equilibrates with NVT, runs the NVE replicas in parallel, and takes the IR spectrum from the Fourier transform of the dipole autocorrelation function. A single minimum is a poor model of the experiment: at 300 K the peptide samples helical and non-helical conformers, and the band shapes report on that ensemble.

In [ ]:
for label, potential in POTENTIALS.items():
    mars(
        f"ir {XYZ} {potential} --charge 1 --md --nvt-time {NVT_PS} --nve-time {NVE_PS} "
        f"--md-timestep {TIMESTEP_FS} --n-replicas {N_REPLICAS} -o md_{TAGS[label]}.dat"
    )

## 3. Compare with experiment

Computed wavenumbers are scaled by 0.97 (as in the paper) and smoothed with a Gaussian (σ = 8 cm$^{-1}$). Every spectrum is normalised in the 1300–1800 cm$^{-1}$ window. For each spectrum we report the amide I and amide II maxima and the Pearson correlation with experiment.

In [ ]:
SCALE = 0.97
SIGMA = 8.0  # cm-1
WINDOW = (1300.0, 1800.0)
AMIDE_I = (1600.0, 1750.0)
AMIDE_II = (1450.0, 1600.0)
GRID = np.arange(WINDOW[0] - 100, WINDOW[1] + 100, 1.0)


def load_mars(path):
    """Read a `mars ir` output on a 1 cm-1 grid: MD spectra are interpolated, harmonic sticks binned."""
    x, y = np.loadtxt(path, comments="#", unpack=True)
    x = x * SCALE
    if np.allclose(np.diff(x), np.diff(x)[0], rtol=1e-3):
        y = np.interp(GRID, x, y)
    else:
        y = np.histogram(x, bins=np.append(GRID, GRID[-1] + 1.0), weights=y)[0]
    return GRID, gaussian_filter1d(y, SIGMA)


def normalise(x, y):
    mask = (x >= WINDOW[0]) & (x <= WINDOW[1])
    y = y - y[mask].min()
    return y / y[mask].max()


def band_max(x, y, band):
    mask = (x >= band[0]) & (x <= band[1])
    return x[mask][np.argmax(y[mask])]


rows = [line.split(",") for line in open(EXPERIMENT) if line[0].isdigit()]
x_exp, y_exp = np.array(rows, dtype=float).T
y_exp = normalise(x_exp, y_exp)
in_window = (x_exp >= WINDOW[0]) & (x_exp <= WINDOW[1])

spectra = {}
print(f"{'':28s}{'amide I':>9s}{'amide II':>10s}{'Pearson r':>11s}")
print(f"{'experiment':28s}{band_max(x_exp, y_exp, AMIDE_I):9.0f}{band_max(x_exp, y_exp, AMIDE_II):10.0f}")
for kind in ("md", "harm"):
    for label, tag in TAGS.items():
        x, y = load_mars(f"{kind}_{tag}.dat")
        y = normalise(x, y)
        r = np.corrcoef(y_exp[in_window], np.interp(x_exp[in_window], x, y))[0, 1]
        spectra[kind, label] = (x, y, r)
        name = f"{label}, {'MD' if kind == 'md' else 'harmonic'}"
        print(f"{name:28s}{band_max(x, y, AMIDE_I):9.0f}{band_max(x, y, AMIDE_II):10.0f}{r:11.2f}")

In [ ]:
colors = {"SO3LR Tauri-M": "#3b6e8f", "ff19SB": "#a4473d"}
fig, axes = plt.subplots(2, 2, figsize=(10, 5), sharex=True, sharey=True)
for col, kind in enumerate(("md", "harm")):
    for row, label in enumerate(TAGS):
        ax = axes[row, col]
        x, y, r = spectra[kind, label]
        ax.fill_between(x_exp, y_exp, color="0.82", lw=0, label="Experiment")
        ax.plot(x_exp, y_exp, color="0.45", lw=0.8)
        ax.plot(x, y, color=colors[label], lw=1.6, label=label)
        ax.text(0.02, 0.88, f"r = {r:.2f}", transform=ax.transAxes, fontsize=8, color=colors[label])
        for band in (AMIDE_I, AMIDE_II):
            ax.axvspan(*band, color="0.5", alpha=0.06, lw=0)
        ax.set_yticks([])
        ax.set_ylim(0, 1.12)
        ax.legend(frameon=False, fontsize=8, loc="upper right")
    axes[0, col].set_title("MD (dipole autocorrelation)" if kind == "md" else "Harmonic, one minimum", fontsize=10)
    axes[1, col].set_xlabel(r"Wavenumber (cm$^{-1}$)")
axes[0, 0].set_xlim(*WINDOW)
fig.supylabel("Intensity (normalised)", fontsize=9)
fig.tight_layout()
fig.savefig("ir_comparison_notebook.png", dpi=200)
plt.show()

## What to look for

With the default settings, the MD spectrum of SO3LR Tauri-M places the amide I and amide II bands within about 5 cm$^{-1}$ of experiment (1691 and 1518 vs 1693 and 1523 cm$^{-1}$, r ≈ 0.96), including the weak carboxylic-acid C=O stretch near 1780 cm$^{-1}$. ff19SB splits the amide I band into two peaks on either side of the experimental maximum and shifts amide II down by about 60 cm$^{-1}$, as reported for ff19SB in the paper. The harmonic spectra of a single minimum miss the band shapes with both potentials, because those shapes come from the conformational ensemble at 300 K.

Your numbers will vary slightly from run to run, because the MD replicas start from random velocities.

## Going further

- More replicas (`N_REPLICAS`) and longer NVE (`NVE_PS`) give smoother spectra; the paper averages 20 × 100 NVE runs.
- `mars ala5lysh.xyz --so3lr-model so3lr-tauri-m --charge 1` searches conformers, and `mars ir ensemble.xyz --all-conformers` gives a Boltzmann-weighted harmonic spectrum.
- MARS also reads CHARMM, OPLS and AMOEBA parameters (`--potential charmm/opls/amoeba`).

## Cite

Please cite SO3LR as described in [How to cite](https://so3lr.github.io/cite/) (with the dated model, here `so3lr-tauri-m-2606`), and:

- MARS: S. Suárez-Dou, M. Gallegos, F. N. Brünig, A. Tkatchenko, *J. Chem. Theory Comput.* **22**, 11125 (2026), [doi:10.1021/acs.jctc.6c01588](https://doi.org/10.1021/acs.jctc.6c01588);
- S. Suárez-Dou, M. Gallegos et al., *Nat. Commun.* (2026), [doi:10.1038/s41467-026-79241-x](https://doi.org/10.1038/s41467-026-79241-x);
- M. Rossi et al., *J. Phys. Chem. Lett.* **1**, 3465 (2010), [doi:10.1021/jz101394u](https://doi.org/10.1021/jz101394u);
- ff19SB: C. Tian et al., *J. Chem. Theory Comput.* **16**, 528 (2020), [doi:10.1021/acs.jctc.9b00591](https://doi.org/10.1021/acs.jctc.9b00591).